# Week 4, Day 2: Describing and Interrogating Data

## Today

By the end of this session you will be able to:

- Read a dataset's `shapeType`, spatial reference (`.name` and `.factoryCode`), extent, and full field list from `Describe`.
- Loop `arcpy.ListFeatureClasses()` across a workspace to build a one-glance inventory of everything in it.
- Read attribute values directly with `arcpy.da.SearchCursor`

## Never trust a file you haven't Described

Before you run a tool against a dataset or write a value from it into a report, it's important to **identify what the dataset actually is.** Not assume. Not guess. Not "there should be a `NAME` field", and it's probably named `NAME`. Assumptions are one keystroke away from a debugging session that `Describe` would have prevented.

***Today, we're using data from the `/data/part1/` directory***

In [ ]:
import arcpy

my_workspace = "C:/Users/pbitterm/Desktop/week0402/data/part1"
arcpy.env.workspace = my_workspace
arcpy.env.overwriteOutput = True

**Expected result?** As always, nothing prints

## `Describe`, a bit deeper

We previously used `Describe` to access the `dataType`, `shapeType`, and `catalogPath` properties. Today, we're go further at where a dataset sits on the map (`extent`), what coordinate system it's in (`spatialReference`), and its complete field schema.

**Predict before you run:** Ohio spans roughly 38–42 degrees north latitude and 80–85 degrees west longitude. Guess whether `ohio_counties.shp`'s `extent` (its `XMin`, `YMin`, `XMax`, `YMax`) falls inside that range. Then guess what `spatialReference.factoryCode` prints, given that this is the same NAD83 dataset from the format "zoo."

In [ ]:
counties_desc = arcpy.da.Describe("ohio_counties.shp")

extent = counties_desc["extent"]
sr = counties_desc["spatialReference"]

print(f"shapeType: {counties_desc['shapeType']}")
print(f"spatial reference name: {sr.name}")
print(f"spatial reference factory code: {sr.factoryCode}")
print(f"extent: {extent.XMin:.2f}, {extent.YMin:.2f} to {extent.XMax:.2f}, {extent.YMax:.2f}")

**Expected result:**

```
shapeType: Polygon
spatial reference name: GCS_North_American_1983
spatial reference factory code: 4269
extent: -84.82, 38.40 to -80.52, 42.33
```

`factoryCode` is the numeric EPSG code behind the name (`4269` is NAD83 geographic). You would actually check the `factoryCode` in a Python script, since string-matching on `.name` is more fragile than comparing an integer. 

The extent confirms the geography: Ohio's counties span from about 84.8°W to 80.5°W, and 38.4°N to 42.3°N, which is the state's real footprint

## Looping through fields

`Describe`'s `fields` property (or `arcpy.ListFields()` directly, which we used in week 3) provides the complete schema — (every field's name, type, and length) without opening the attribute table

**Predict before you run:** `ohio_counties.shp` has 18 of its own attribute fields, plus the two arcpy always adds. Every text field in this particular shapefile happens to share the exact same declared length. Why might that be?

In [ ]:
fields = arcpy.ListFields("ohio_counties.shp")

for field in fields:
    print(f"{field.name:<12} {field.type:<10} length={field.length}")

**Expected result:** one line per field, `FID` and `Shape` first, then the 18 attribute fields in file order: `STATEFP`, `COUNTYFP`, `COUNTYNS`, `GEOID`, `GEOIDFQ`, `NAME`, `NAMELSAD`, `LSAD`, `CLASSFP`, `MTFCC`, `CSAFP`, `CBSAFP`, `METDIVFP`, `FUNCSTAT`, `ALAND`, `AWATER`, `INTPTLAT`, `INTPTLON`. 

Every text field (`NAME` included) reports `type=String length=80`, even though no county name in this file comes anywhere close to 80 characters. 

`ALAND` and `AWATER` (land and water area, in square meters) report `type=Double length=18`. 

The mismatch between "declared length" and "actual content" is normal for a shapefile: a DBF field's width is a *ceiling* set when the file was built, not a measurement of what's actually stored. Yet another weird shapefile quirk

**Your turn:** Run the same `extent`/`spatialReference` block from above against `portage_munis.shp` instead of `ohio_counties.shp`. It should be the same NAD83 coordinate system, so you predict the `factoryCode` before you check. Then loop `arcpy.ListFields()` on `portage_munis.shp` and compare its declared field lengths to `ohio_counties.shp`'s uniform `length=80` text fields... are they the same?


In [ ]:
## Try it here



## `Describe` on a non-spatial file

`Describe` isn't only for feature classes. Point it at a plain table such as `counties_attrs.csv`, which you can find in `formats_zoo`. It still provides an answer, just of a different kind.

**Predict before you run:** `counties_attrs.csv` holds the same attribute columns as the counties dataset, but no geometry at all. Guess what `dataType` comes back as, and whether `Describe` even has a `shapeType` to report this time.

In [ ]:
csv_desc = arcpy.da.Describe("YOURPATHHERE/data/part1/formats_zoo/counties_attrs.csv")

print(f"dataType: {csv_desc['dataType']}")
print("shapeType" in csv_desc)

**Expected result:** `dataType: TextFile`, then `False`. 

A `.csv` describes as a text file, not a feature class. There is  no geometry, so no `shapeType` key exists in the dictionary at all, rather than existing and being empty. 

### `Describe` on a GeoJSON

We previously looked at four formats (i.e., shapefile, file geodatabase, GeoPackage, GeoParquet), each of which `Describe` treats as full-fledged datasets. 

`streams_portage.geojson` (in the `part1` folder), is a fifth format. `Describe` can describe it, same as any shapefile.

**Predict before you run:** `streams_portage.geojson` never showed up in `ListFeatureClasses()`'s output, which we confirmed last week. Given that, do you expect `arcpy.da.Describe("streams_portage.geojson")` to raise an error the way `csv_desc["shapeType"]` would have without the `in` guard above, or does it succeed and report a `dataType` much like `ohio_counties.shp` did?

In [ ]:
## Before running, make sure you have your workspace setup properly and that the file `streams_portage.geojson` is in that workspace. Then run the following code to describe it.

geojson_desc = arcpy.da.Describe("streams_portage.geojson")

print(f"dataType: {geojson_desc['dataType']}")
print(f"shapeType: {geojson_desc['shapeType']}")
print(f"spatial reference: {geojson_desc['spatialReference'].name}")
print([f.name for f in geojson_desc['fields']])

**Expected result:**

```
dataType: FeatureClass
shapeType: Polyline
spatial reference: GCS_WGS_1984
```

plus a `fields` list with five entries: `osm_id`, `name`, `waterway`. These are the GeoJSON's own three attributes, straight out of its `"properties"` block, in addition to arcpy's own auto-generated OID field and `Shape`. 


## Inventory your data

If we combine `ListFeatureClasses()` and `Describe`, we can construct a summary of everything in a workspace, without every touching ArcGIS Pro's catalog view

**Predict before you run:** The main `part1` folder holds three feature classes (`ohio_counties.shp`, `ohio_state_parks.shp`, `portage_munis.shp`) all of which are polygons and in a CRS we've seen before. Predict whether the loop below prints all three with the same `spatialReference.name`, or whether one of them differs.

In [ ]:
for fc in arcpy.ListFeatureClasses():
    desc = arcpy.da.Describe(fc)
    field_count = len(desc["fields"])
    print(f"{fc:<22} {desc['shapeType']:<10} {desc['spatialReference'].name:<28} {field_count} fields")

**Expected result:** three lines, one per feature class, all reporting `Polygon`. 

But do the coordinate systems don't match?

`ohio_counties.shp` and `portage_munis.shp` report `GCS_North_American_1983` (NAD83, the same as before ), but `ohio_state_parks.shp` reports `GCS_WGS_1984`. This is a different geographic coordinate system entirely, inherited from wherever that dataset originally came from. 

This is precisely why NEVER TRUST A DATASET WE HAVEN'T DESCRIBED. Nothing about opening these three files in ArcGIS Pro's map view would have flagged the mismatch, since the software reprojects on the fly for display.



**Your turn:** Extend the inventory loop's f-string to also print `desc['dataType']` for each feature class. Predict first: will all three print the same `dataType` you saw for `counties.shp` last class, or does the container matter here too, given all three of today's files are plain shapefiles?



In [ ]:
## Try it here



## Cursors, in brief

Every `print()` command you've run so far has told you about a dataset's *structure*. Nothing has shown you actual row values, like "what's the third county's name." 

To do that, arcpy has **cursors**. These are objects that step through a table or feature class one row at a time. `arcpy.da.SearchCursor` is the read-only one, and it's the only cursor we're going to bother with.

**Predict before you run:** The cursor below reads two fields, `NAME` and `SHAPE@AREA`. The latter is a **geometry token**, arcpy's way of pulling a calculated geometry property instead of a stored field. 

It reads these from `ohio_counties.shp`, and prints the first 5 rows. 

Predicts the printed output. Do we expect one clean number per county, or something odd, given that this shapefile's coordinate system is geographic (degrees), not projected (feet or meters).

In [ ]:
with arcpy.da.SearchCursor("ohio_counties.shp", ["NAME", "SHAPE@AREA"]) as cur:
    for i, row in enumerate(cur):
        if i >= 5:
            break
        print(row)

        

**Expected result:** 5 tuples, each `(county_name, area_value)` — something like `('Hancock', 0.1479...)`, `('Allen', 0.1124...)`, and so on, in the shapefile's internal row order (not alphabetical — nothing sorted these). 

A tuple is a data structure like a list, but it is immutable (cannot be changed). You can access its elements by index, just like a list. For example, row[0] would give you the NAME and row[1] would give you the SHAPE@AREA.

The area numbers are the odd part your prediction should have flagged: `ohio_counties.shp` is in a geographic coordinate system (NAD83, degrees), so `SHAPE@AREA` hands back square *degrees*, not square miles.

In [ ]:
with arcpy.da.SearchCursor("ohio_counties.shp", ["NAME", "SHAPE@AREA"]) as cur:
    for i, row in enumerate(cur):
        if i >= 5:
            break
        print(f"County: {row[0]}, area: {row[1]}")

**Row access when you must.** `SearchCursor` is the tool for the (fairly uncomment) times you need to work with individual features (rows) in a dataset. This course intentionally does not cover `UpdateCursor` or `InsertCursor` (arcpy's row-*writing* cursors), and we'll do more of that in part 2 of the course when we start using GeoPandas.

**Your turn:** Change the cursor's break condition from `i >= 5` to `i >= 3`, and point it at `streams_portage.geojson` instead of `ohio_counties.shp`, swapping `SHAPE@AREA` for `SHAPE@LENGTH`. 

Predict first, using the `spatialReference` you just found for the GeoJSON: will the length values look like plausible real-world distances, or the same kind of small, unit-mismatched fraction `SHAPE@AREA` gave you above?

In [ ]:
## Try it here



## For next time

- Lab 02 starts today
- Sketch 2 due soon, peer reviews follow
- Readings posted to Canvas

**Next week:** turning repeated code into functions — parameters, return values, and the start of writing arcpy code other people (and future you) can (re)use